In [18]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go


# Read the dataset
df = pd.read_csv('../Data_age_research/2_Mobility_per_person_by_transport_mode_and_purpose_(84710NED).csv',
    sep=";",
    skiprows=5,
    header=[0, 1, 2, 3],
    decimal=",",
    na_values=[".", ""]
)

# Select each transport mode for the total population
# and all travel purposes
columns = [
    col for col in df.columns
    if col[0] == "Total"
    and col[1] != "Total"
    and col[2] == "Total"
    and col[3] == "number"
]

# Extract the years
years = pd.to_numeric(
    df.iloc[:, 0].astype(str).str.replace("*", "", regex=False),
    errors="coerce"
)

# Keep the years 1999–2025
mask = years.between(1999, 2025)


fig = go.Figure()

for col in columns:
    fig.add_trace(
        go.Scatter(
            x=years[mask],
            y=pd.to_numeric(df.loc[mask, col], errors="coerce"),
            mode="lines+markers",
            name=str(col[1]),
            hovertemplate=(
                "Year: %{x}<br>"
                "Transport mode: " + str(col[1]) + "<br>"
                "Average trips per person per day: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )

fig.update_layout(
    title="Changes in Transport Mode Use in the Netherlands (1999–2025)",
    xaxis_title="Year",
    yaxis_title="Average trips per person per day",
    hovermode="closest",
    template="plotly_white"
)

fig.update_xaxes(
    range=[1999, 2025],
    dtick=2
)
fig.write_image("../Output/absolute_changes_in_transport_mode.png")
fig.show()


In [ ]:
def calculate_shares(data):
    """
    Calculate the percentage share of each transport mode per year.
    """
    total_trips_per_year = data.sum(axis=1)
    shares = data.div(total_trips_per_year, axis=0) * 100

    return shares

In [ ]:
# Select the data for the selected transport modes
data = df.loc[mask, columns].copy()

# Convert values to numeric
data = data.apply(pd.to_numeric, errors="coerce")

# Give each column the transport mode name
data.columns = [col[1] for col in columns]

# Set the year as the index
data.index = years.loc[mask].astype(int)
data.index.name = "Year"

# Calculate percentage shares
shares = calculate_shares(data)

# Round percentages
shares = shares.round(2)

# Turn the index into a Year column
shares = shares.reset_index()

# Display the clean table
display(shares)

,Year,Car (driver),Car (passenger),Train,Bus/tram/metro,Bicycle,Walking,Other mode of transport
0,1999,30.88,14.12,2.06,4.71,25.59,19.12,3.53
1,2000,31.23,13.81,2.10,4.50,25.83,18.92,3.60
2,2001,31.21,13.64,2.12,4.55,25.76,19.09,3.64
3,2002,31.71,13.41,2.13,4.27,25.91,18.90,3.66
4,2003,31.61,13.37,2.13,4.26,26.44,18.84,3.34
5,2004,31.61,13.37,2.13,4.26,26.44,18.84,3.34
6,2005,31.15,13.08,2.18,4.05,26.48,19.63,3.43
7,2006,30.96,12.69,2.17,4.02,26.63,20.12,3.41
8,2007,31.56,12.50,2.19,4.06,26.56,19.69,3.44
9,2008,31.23,12.30,2.21,4.10,26.81,19.87,3.47


In [17]:
# Create a 100% stacked bar chart
fig = go.Figure()

# Select transport mode columns (exclude Year)
transport_modes = [col for col in shares.columns if col != "Year"]

for mode in transport_modes:
    fig.add_trace(
        go.Bar(
            x=shares["Year"],
            y=shares[mode],
            name=mode,
            hovertemplate=(
                "Year: %{x}<br>"
                "Transport mode: " + str(mode) + "<br>"
                "Share of trips: %{y:.2f}%"
                "<extra></extra>"
            )
        )
    )

# Configure the chart
fig.update_layout(
    title="Changes in Transport Mode Shares in the Netherlands (1999–2025)",
    xaxis_title="Year",
    yaxis_title="Share of trips (%)",
    barmode="stack",
    template="plotly_white",
    hovermode="closest",
    legend_title="Transport mode"
)

# Set y-axis from 0% to 100%
fig.update_yaxes(
    range=[0, 100],
    dtick=10,
    ticksuffix="%"
)

# Display every second year to keep the x-axis readable
fig.update_xaxes(
    tickmode="linear",
    dtick=2
)
fig.write_image("../Output/%change_in_transport_mode_shares_100%chart.png")
fig.show()

In [16]:
# Create an interactive line chart
fig = go.Figure()

# Select transport mode columns (exclude Year)
transport_modes = [col for col in shares.columns if col != "Year"]

for mode in transport_modes:
    fig.add_trace(
        go.Scatter(
            x=shares["Year"],
            y=shares[mode],
            mode="lines+markers",
            name=mode,
            line=dict(width=2),
            marker=dict(size=4),
            hovertemplate=(
                "Year: %{x}<br>"
                "Transport mode: " + str(mode) + "<br>"
                "Share of trips: %{y:.2f}%"
                "<extra></extra>"
            )
        )
    )

# Configure the chart
fig.update_layout(
    title="Changes in Transport Mode Shares in the Netherlands (1999–2025)",
    xaxis_title="Year",
    yaxis_title="Share of trips (%)",
    template="plotly_white",
    hovermode="closest",
    legend_title="Transport mode"
)

# Set y-axis from 0% to 100%
fig.update_yaxes(
    range=[0, 100],
    dtick=10,
    ticksuffix="%"
)

# Display every second year
fig.update_xaxes(
    tickmode="linear",
    dtick=2
)
fig.write_image("../Output/%change_in_transport_mode_shares_line_chart.png")
fig.show()